# Preprocessing wondr by BNI

**Tahap:** 1-5 (Load → Filter Rating → Relative Time → Dedup → Text & Slang Normalization)

**Input:** `data/raw/wondr_by_BNI_raw.csv` (44,627 reviews)
**Output:** `data/processed/wondr_preprocessed.csv` (~10K negative reviews, normalized)

**Metodologi:**
- Filter rating 1-2 sebagai proxy keluhan teknis
- Relative month (1-12) dari launch date 5 Juli 2024
- Slang normalization: Salsabila lexicon + banking extension

In [1]:
%load_ext autoreload
%autoreload 2

import pandas as pd
from datetime import datetime

from utils.preprocessing import (
    load_raw_reviews,
    filter_negative_ratings,
    add_relative_time_columns,
    drop_exact_duplicates,
    apply_normalization,
    load_slang_dict,
    apply_slang_normalization,
    filter_short_reviews,
    save_preprocessed_outputs,
)

# Configure pandas display
pd.set_option("display.max_colwidth", 100)
pd.set_option("display.width", 150)

## Tahap 1: Load Raw Reviews & Filter Rating 1-2

Filter rating 1-2 sebagai proxy keluhan teknis. Rating 3-5 di-drop dari analisis.

In [2]:
df = load_raw_reviews("data/raw/wondr_by_BNI_raw.csv")
df = filter_negative_ratings(df)

✅ Loaded 44,627 reviews from data/raw/wondr_by_BNI_raw.csv
✅ Rating filter: kept 10,319 / 44,627 reviews (dropped 34,308 non-negative reviews).
   Distribution: {1: 8313, 2: 2006}


## Tahap 2: Ekstraksi Relative Month & Week

- Launch date wondr: 5 Juli 2024
- Window: 12 bulan (5 Jul 2024 - 4 Jul 2025)
- Relative month dihitung calendar-based (5 Jul - 4 Aug = bulan 1, dst)

In [3]:
WONDR_LAUNCH = datetime(2024, 7, 5)
df = add_relative_time_columns(df, WONDR_LAUNCH)

✅ Relative time extraction (launch=2024-07-05, window=12 months):
   Kept 10,319 / 10,319 reviews in window.
   Distribution per relative_month: {1: 327, 2: 431, 3: 595, 4: 769, 5: 3005, 6: 1487, 7: 809, 8: 713, 9: 552, 10: 439, 11: 457, 12: 735}


## Tahap 3: Hapus Exact Duplicates

Dedup berdasarkan `review_text` (raw, sebelum normalisasi). Keep first
(earliest timestamp). Top 5 duplicates di-log untuk sanity check.

In [4]:
df = drop_exact_duplicates(df)

✅ Exact duplicate removal (by 'review_text', keep first):
   Kept 10,131 / 10,319 reviews (dropped 188 duplicates).

   Top 5 most-duplicated review texts:
     1. [12x] 'mantap'
     2. [12x] 'Sering error'
     3. [11x] 'Ribet'
     4. [11x] 'baik'
     5. [11x] 'ok'


## Tahap 4: Text Normalization

Pipeline 8 step: lowercase → URL → emoji → number → repeat collapse → 
punctuation → single chars → whitespace.

**Catatan:** Stopword removal TIDAK dilakukan di sini (best practice
BERTopic — stopwords di-handle di tahap c-TF-IDF).

In [5]:
df = apply_normalization(df)

✅ Text normalization applied to 10,131 reviews.
   Empty after cleaning: 4 reviews (will be dropped at Stage 7 short-review filter).
   Word count (after) — mean: 17.7, median: 13, max: 91


## Tahap 5: Slang Normalization

Sumber kamus:
1. **Salsabila lexicon** (Salsabila et al. 2018) — base lexicon Bahasa Indonesia colloquial
2. **Banking extension** — 39 entries domain-specific (compiled manual)

**Filter applied:**
- 348 degrading reduplication mappings di-drop (mis. 'baru' → 'baru baru')
- 71 entri Salsabila di-blocklist untuk 6 kata baku Indonesia
  (`apa`, `nya`, `sekali`, `bener`, `minta`, `bukan`) yang ter-corrupt
  setelah preprocessing — diidentifikasi via data-driven inspection
  Top 30 most-triggered mappings.

In [6]:
slang_dict = load_slang_dict(
    salsabila_path="dictionaries/salsabila.csv",
    banking_ext_path="dictionaries/banking_extension.csv",
    # blocklist defaults to INDONESIAN_PROTECTED_WORDS in module
)

df = apply_slang_normalization(df, slang_dict)

   Dropped 348 degrading reduplication mappings (e.g., 'baru' → 'baru baru').
   Dropped 71 blocklisted entries (Indonesian standard words protected from mis-replacement).
✅ Loaded Salsabila lexicon: 3,641 entries (after preprocessing & dedup).
✅ Loaded banking extension: 65 entries.
✅ Merged dictionary: 3,682 total entries (24 banking entries override Salsabila).
✅ Slang normalization applied to 10,131 reviews.
   Reviews with at least one slang replacement: 7,724 (76.2%).
   Word count (after) — mean: 17.9, median: 14, max: 91


## Inspeksi Hasil


In [7]:
# Sample 10 review acak — sebelum vs sesudah preprocessing
df[["review_text", "review_text_cleaned", "word_count_after",
    "relative_month", "rating"]].sample(10, random_state=42)

,review_text,review_text_cleaned,word_count_after,relative_month,rating
3780,"mau buat transaksi gak bisa . kecewa banget pake aplikasi ini , bank gede kok bisa2nya jam segin...",mau buat transaksi tidak bisa kecewa banget pakai aplikasi ini bank gede kok bisa nya jam segini...,30,5,1
532,Susah bgt verifikasi wajah ribet asli,susah banget verifikasi wajah ribet asli,6,2,1
1155,Halah.. sama saja pun.. malah lebih lengkap bni mobile banking.. ini modus cari nasabah aja.. ny...,halah sama saja pun malah lebih lengkap bni mobile banking ini modus cari nasabah saja nyesal pu...,45,3,1
1953,hapus saja gk guna. ribet pake foto wajah segala foto ktp mash mudah skrg uda tua jelas lain. gk...,hapus saja tidak guna ribet pakai foto wajah segala foto ktp mash mudah mudahan sekarang sudah t...,22,4,1
4924,"Ribet, mau ngecek riwayat ae susah, enak BNI Banking malah ganti, mau beli token listrik aja sus...",ribet mau cek riwayat saja susah enak bni banking malah ganti mau beli token listrik saja susah ...,21,5,1
4238,hari ini ada apa aplikasi wondr.... ko tidak bisa login...,hari ini ada apa aplikasi wondr kok tidak bisa login,10,5,1
3038,Aplikasinya eror mulu ga pernah perjalan normal,aplikasi error terus tidak pernah perjalan normal,7,5,1
1441,Aplikasi Ampas apa apan di pixel 7 gabisa verif. padahal wajah udah sesuai kamera mumpuni malah ...,aplikasi ampas apa apan di pixel enggak bisa verifikasi padahal wajah sudah sesuai kamera mumpun...,19,4,1
1373,aplikasi paling jelek yg pernah ada daftar nya pun ribet udh tau itu muka kita malah di ulang²,aplikasi paling jelek yang pernah ada daftar nya pun ribet sudah tau itu muka kita malah di ulang,18,4,1
6870,"Ini banyak fitur yg gk ada seperti tarik tunai tanpa kartu, buat bayar kuliah di uin Jakarta jug...",ini banyak fitur yang tidak ada seperti tarik tunai tanpa kartu buat bayar kuliah di uin jakarta...,23,7,1


In [8]:
# Distribusi review per relative_month
print("Distribusi review per relative_month:")
print(df["relative_month"].value_counts().sort_index())
print(f"\nTotal reviews after Tahap 1-5: {len(df):,}")

Distribusi review per relative_month:
relative_month
1      327
2      431
3      592
4      762
5     2963
6     1465
7      794
8      694
9      531
10     418
11     437
12     717
Name: count, dtype: int64

Total reviews after Tahap 1-5: 10,131


## Tahap 6: Filter Short Reviews (<5 kata)

Drop reviews dengan word count <5 setelah Stage 4-5 normalization.
Diletakkan setelah slang normalization karena slang dapat expand short
forms (`mbanking` → `mobile banking`), mengubah word count.

Diletakkan **sebelum** Stage 7 (language filter) berdasarkan audit:
fasttext lid.176 punya akurasi rendah pada text <5 kata.

In [9]:
df = filter_short_reviews(df, min_words=5)

✅ Short review filter (min_words=5):
   Kept 8,982 / 10,131 reviews (dropped 1,149 reviews with < 5 words).
   Word count (after) — mean: 19.8, median: 15, max: 91


## Tahap 7: Language Filtering — DEPRECATED

Language filter (fasttext lid.176) **tidak digunakan** di pipeline final
berdasarkan data-driven audit:

- Iterasi 1: Tier 3 = 471 (4.6%), 5/5 sampel = false positive (Indonesian)
- Iterasi 2 (setelah reorder Stage 6 ↔ 7): Tier 3 = 123 (1.4%), tetap 9/10 false positive
- Pattern misclassification: banking loanwords → English, Indonesian-Malay similarity, typo

**Conclusion:** fasttext lid.176 punya systemic bias di domain banking
Indonesia. Asumsi domain (>95% Indonesian) dipertahankan; outlier
non-Indonesian akan ke-cluster sebagai outlier topic di BERTopic.

Detail audit: lihat `99_dev_wondr.ipynb` dan `preprocessing_methodology.md`.

## Tahap 8: Save Outputs (BERTopic-ready & Full Audit)

Save dua versi:
- **`wondr_bertopic.csv`** (slim) — kolom essential untuk pipeline BERTopic
- **`wondr_full.csv`** (audit) — semua kolom + metadata preprocessing

In [10]:
# Update import — tambahkan filter_short_reviews & save_preprocessed_outputs
# (atau pastikan udah ada di Cell 2 import)

save_preprocessed_outputs(
    df,
    bertopic_path="data/processed/wondr_bertopic.csv",
    full_path="data/processed/wondr_full.csv",
)

✅ BERTopic-ready saved: data/processed/wondr_bertopic.csv
   Shape: (8982, 6), Columns: ['review_id', 'review_text_cleaned', 'relative_month', 'relative_week', 'date_wib', 'rating']

✅ Full audit saved: data/processed/wondr_full.csv
   Shape: (8982, 20), Columns: 20 columns


## Validasi Akhir

In [11]:
# Validasi 1: Distribusi review per relative_month
print("Distribusi review per relative_month:")
print(df["relative_month"].value_counts().sort_index().to_string())
print(f"\nTotal final reviews: {len(df):,}")
print(f"Mean per month: {len(df) / 12:.0f}")

Distribusi review per relative_month:
relative_month
1      302
2      395
3      544
4      678
5     2568
6     1297
7      727
8      621
9      462
10     357
11     388
12     643

Total final reviews: 8,982
Mean per month: 748


In [12]:
# Validasi 2: Sample 30 review (manual visual check)
sample = df.sample(30, random_state=42)
for _, row in sample.iterrows():
    print(f"[M{row['relative_month']:02d} R{row['rating']}] {row['review_text_cleaned'][:100]}")

[M09 R1] makin susah dibuka sekarang tolong perbaiki aplikasi ini min
[M01 R1] baru install wondr sempat berhasil login tapi lupa password setelah itu verifikasi ktp tapi selalu g
[M03 R1] susah banget pas vermuknya sudah berkali coba tetap gagal terus woi
[M12 R2] katanya kartu atm bakal dikirim dirumah tapi nyatanya tidak sudah berbulan tidak dikirim dikirim pad
[M12 R1] sayang saldo mengendap yang harus disisakan cukup besar sebesar ribu terima kasih atas pelayan yang 
[M09 R1] kalau sudah jam kenapa tidak bisa transfer ris dan lain kalau begini terusending caring bank lain ca
[M05 R1] habis update kok tidak bisa dipakai aplikasi
[M08 R1] saat transaksi masih proses dan saldo terpotong sangat lama untuk memeriksa nya harus menunggu hk
[M09 R1] topup dana tidak masuk woy
[M03 R1] aplikasi tidak bisa di pakai tolong diperbaiki buat kelancaran nasabah masak mau masuk saja ribet be
[M07 R1] kenapa pas verifikasi wajah gagal terus padahal pencahayaan bagus sudah mengikuti petunjuk tapi t

In [13]:
# Validasi 3: Top 100 most frequent words (untuk iterative refine banking dict)
from collections import Counter

all_words = " ".join(df["review_text_cleaned"].fillna("")).split()
word_counts = Counter(all_words)

print(f"Total unique words: {len(word_counts):,}")
print(f"Total tokens: {sum(word_counts.values()):,}")
print(f"\nTop 100 most frequent words:")
for i, (word, count) in enumerate(word_counts.most_common(100), 1):
    print(f"  {i:3d}. {word:25s} ({count:,})")

Total unique words: 8,021
Total tokens: 177,685

Top 100 most frequent words:
    1. tidak                     (6,513)
    2. aplikasi                  (5,361)
    3. bisa                      (4,102)
    4. di                        (3,727)
    5. sudah                     (2,877)
    6. ini                       (2,856)
    7. bni                       (2,765)
    8. yang                      (2,491)
    9. saya                      (2,227)
   10. nya                       (2,034)
   11. terus                     (1,973)
   12. ada                       (1,897)
   13. mau                       (1,868)
   14. lagi                      (1,824)
   15. saja                      (1,713)
   16. dan                       (1,679)
   17. wondr                     (1,611)
   18. transaksi                 (1,448)
   19. verifikasi                (1,436)
   20. mobile                    (1,434)
   21. pakai                     (1,416)
   22. tapi                      (1,403)
   23. ke           